In [2]:
! pip install langchain-text-splitters


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Set display options for Pandas so we can read long reviews in the notebook
pd.set_option('display.max_colwidth', None)

c:\Users\shash\OneDrive\Desktop\Data\nlp_review_insights\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Cell 2: Data Loading & MVP Filtering (Multi-file)
import pandas as pd

# 1. Define the exact file paths from your data folder
file1 = '../data/1429_1.csv'
file2 = '../data/Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products.csv'
file3 = '../data/Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products_May19.csv'

print("Loading datasets... this might take a few seconds.")

# 2. Load all three datasets (low_memory=False prevents warnings on large mixed-type CSVs)
df1 = pd.read_csv(file1, low_memory=False)
df2 = pd.read_csv(file2, low_memory=False)
df3 = pd.read_csv(file3, low_memory=False)

# 3. Combine them into one single dataframe
df_combined = pd.concat([df1, df2, df3], ignore_index=True)

# 4. Standardize column names for easier handling
df_combined = df_combined.rename(columns={
    'name': 'product_name',
    'reviews.title': 'review_title',
    'reviews.text': 'review_text',
    'reviews.rating': 'rating'
})

# 5. Keep only what we need and drop rows missing critical text
df_clean = df_combined[['product_name', 'review_title', 'review_text', 'rating']].dropna()

# 6. Sample 1,500 rows for our MVP to keep prototyping fast
df_mvp = df_clean.sample(n=1500, random_state=42).reset_index(drop=True)

print(f"Total raw reviews across all 3 files: {len(df_combined)}")
print(f"Cleaned MVP Dataset Shape: {df_mvp.shape}")
print(f"Unique Devices in MVP: {df_mvp['product_name'].nunique()}")

# Peek at the data to ensure it looks right
df_mvp.head(3)

Loading datasets... this might take a few seconds.
Total raw reviews across all 3 files: 67992
Cleaned MVP Dataset Shape: (1500, 4)
Unique Devices in MVP: 55


,product_name,review_title,review_text,rating
0,"Brand New Amazon Kindle Fire 16gb 7 Ips Display Tablet Wifi 16 Gb Blue,,,",Not the greatest for kids,"We purchased this tablet for our 5 y ear old since all the other kid-friendly tablets were limited it what they had on them. Something should be changed regarding the port for the charger. The little area with the small bar in it keeps bending to the point, 5 months later it broke off and of course the warranty has ended. I am over tablets. If you can buy it on black Friday for 25.00 bucks or less, it should be that price all year long. So now he has no tablet because I am not sending it out to have someone else's sent back to me..re-furbished. I am very disappointed.",1.0
1,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes Special Offers, Magenta",Inexpensive tablet,Good inexpensive tablet. I use it to control my E series vizio and andriod box. Apps open slower than on my Galaxy S7 edge but other than that everyting works as expected. Note that google play store is not available without modification,3.0
2,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes Special Offers, Magenta",works good for the price,Works great was a gift for daughter she loves it!!,5.0


In [5]:
# Initialize the text splitter
# 400 characters is usually a sweet spot for a single thought in a review
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=40,
    separators=["\n\n", "\n", ".", "!", "?", " ", ""]
)

documents = []

# Iterate through the dataframe and build Document objects
for index, row in df_mvp.iterrows():
    # Combine title and text for better context
    full_review = f"{row['review_title']} - {row['review_text']}"
    
    # Split the review into smaller chunks
    chunks = text_splitter.split_text(full_review)
    
    # Attach metadata to each chunk
    for chunk in chunks:
        doc = Document(
            page_content=chunk,
            metadata={
                "product_name": row['product_name'],
                "rating": row['rating']
            }
        )
        documents.append(doc)

print(f"Successfully created {len(documents)} chunked documents with metadata.")
# Peek at the first document to verify
print("\nSample Document:")
print(documents[0].page_content)
print(documents[0].metadata)

Successfully created 1624 chunked documents with metadata.

Sample Document:
Not the greatest for kids - We purchased this tablet for our 5 y ear old since all the other kid-friendly tablets were limited it what they had on them. Something should be changed regarding the port for the charger. The little area with the small bar in it keeps bending to the point, 5 months later it broke off and of course the warranty has ended. I am over tablets
{'product_name': 'Brand New Amazon Kindle Fire 16gb 7 Ips Display Tablet Wifi 16 Gb Blue,,,', 'rating': 1.0}


#### Cell 4: Load the Embedding Model

In [6]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# This will download a small (~80MB) model to your local machine the first time you run it.
# It converts sentences into 384-dimensional mathematical arrays.
print("Loading Embedding Model...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
print("Model Loaded!")

Loading Embedding Model...


C:\Users\shash\AppData\Local\Temp\ipykernel_4560\770943671.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6524.60it/s]


Model Loaded!


#### Cell 5: Create and Save the Vector Database

In [7]:
from langchain_community.vectorstores import Chroma
import os

# We will save the database in your data folder so we don't have to rebuild it every time
persist_directory = '../data/chroma_db'
os.makedirs(persist_directory, exist_ok=True)

print(f"Embedding {len(documents)} chunks and saving to ChromaDB...")
print("This might take a minute or two on your local machine...")

# Build the database
vector_db = Chroma.from_documents(
    documents=documents, 
    embedding=embeddings, 
    persist_directory=persist_directory
)

print("✅ Vector Database successfully created and saved locally!")

# Let's do a quick test search to prove it works
test_query = "What do people say about the battery life?"
results = vector_db.similarity_search(test_query, k=2) # Find top 2 matches

print("\n--- TEST SEARCH RESULTS ---")
for res in results:
    print(f"\nProduct: {res.metadata['product_name']}")
    print(f"Text: {res.page_content}")

Embedding 1624 chunks and saving to ChromaDB...
This might take a minute or two on your local machine...
✅ Vector Database successfully created and saved locally!

--- TEST SEARCH RESULTS ---

Product: AmazonBasics AAA Performance Alkaline Batteries (36 Count)
Text: short battery life - short battery life

Product: Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16 GB, Pink Kid-Proof Case
Text: . So far, we really like this tablet. The only thing that I still need time to determine its battery life. After about 2 hours of game play, the battery is about at 50%. Very satisfied.


#### Cell 6: Securely Load the LLM

In [13]:
import os
from getpass import getpass
from langchain_groq import ChatGroq

# Prompt for the API key securely (only if not already set)
if "GROQ_API_KEY" not in os.environ:
    print("Paste your Groq API Key below and hit Enter:")
    os.environ["GROQ_API_KEY"] = getpass()

# Initialize the LLM with the updated model name
llm = ChatGroq(
    model_name="llama-3.1-8b-instant",  # <-- The new, supported model!
    temperature=0
)
print("\n✅ Llama-3.1 Initialized!")


✅ Llama-3.1 Initialized!


#### Cell 7: Build and Test the Complete RAG Pipeline

In [10]:
! pip install langchain-classic


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# 1. Define how the LLM should behave
system_prompt = (
    "You are an expert product analyst. Use the following retrieved customer reviews "
    "to answer the user's question. If the answer cannot be found in the reviews, "
    "simply state that you do not have enough data. Keep your answer concise, "
    "professional, and mention specific product names when applicable.\n\n"
    "Reviews Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# 2. Combine the LLM and the Prompt
document_chain = create_stuff_documents_chain(llm, prompt)

# 3. Create the Retriever (Fetch the top 5 most relevant review chunks)
retriever = vector_db.as_retriever(search_kwargs={"k": 5})

# 4. Create the final end-to-end RAG chain
rag_chain = create_retrieval_chain(retriever, document_chain)
print("✅ Full RAG Pipeline Assembled!\n")

# --- THE GRAND TEST ---
test_question = "What do users say about the battery life? Give me specific product examples."
print(f"🤖 User: {test_question}")
print("⚙️ AI is thinking...\n")

# Run the pipeline
response = rag_chain.invoke({"input": test_question})

# Print the final generated answer
print("📊 AI Analyst Report:")
print("-" * 40)
print(response["answer"])

✅ Full RAG Pipeline Assembled!

🤖 User: What do users say about the battery life? Give me specific product examples.
⚙️ AI is thinking...

📊 AI Analyst Report:
----------------------------------------
Based on the customer reviews, users have mixed opinions about the battery life of different products. 

Some users mention that the battery life is short, such as:

- A Kindle (original) with a battery life of about 8 days.
- A tablet with a battery life of about 3 days (as mentioned in the first review).
- A battery with extremely limited shelf life, going dead in a hurry if not used within 6 months.

However, another user mentions that they like the battery endurance of a product, specifically mentioning a new 8" screen.
